# Visualização Interativa: Distribuição Geográfica de Armadores de Pesca no Brasil

**Disciplina:** Computação Gráfica e Visualização I (INF01047) — INF/UFRGS  
**Laboratório 3**

**Dataset:** Registros de Armadores de Pesca — MPA/SERMOP  
**Formato:** `.xlsx` com separador `;`, colunas: `Nome / Razão Social`, `Vínculo com o RGP`, `Número no RGP`, `cidade`, `UF`, `Tipo de Pessoa`

---

## Estrutura do notebook

| Célula | O que faz |
|--------|-----------|
| 1 | Instala dependências |
| 2 | Importações |
| 3 | **Carrega e limpa o arquivo real** (`armadores_pesca.xlsx`) |
| 4 | Geocodifica municípios (lat/lon) |
| 5 | Pré-calcula estatísticas por estado e região |
| 6 | Baixa GeoJSON dos estados |
| 7 | Prepara dados de visualização (pontos, heatmap, bbox) |
| 8 | **Gera o mapa HTML interativo** (chama `build_map_v3.py`) |
| 9 | Salva e baixa o arquivo HTML |
| 10 | Gráfico estático para o relatório |


## 1. Instalação de dependências

In [ ]:
!pip install pandas openpyxl requests numpy matplotlib geopy --quiet
print("OK")

## 2. Importações

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import time
import os
import warnings
warnings.filterwarnings('ignore')
print("Bibliotecas importadas com sucesso!")

## 3. Carregamento e limpeza do arquivo real

> **Passo a passo:**
> 1. Baixe `armadores_pesca.xlsx` do link do TAREFAS.md
> 2. No Colab: clique no ícone de pasta → Upload → selecione o arquivo
> 3. Execute esta célula

O arquivo tem:
- Separador `;` entre colunas
- Coluna `cidade` no formato `"Belém,PA,Brasil"` — extraímos apenas o município
- Coluna `UF` já contém a sigla do estado (PA, SC, CE…)
- Encoding gerenciado pelo openpyxl (xlsx) ou tentativa automática (csv)


In [ ]:
# ── Mapeamento estado → região ─────────────────────────────────────
REGIOES = {
    'AC':'Norte','AM':'Norte','AP':'Norte','PA':'Norte','RO':'Norte','RR':'Norte','TO':'Norte',
    'AL':'Nordeste','BA':'Nordeste','CE':'Nordeste','MA':'Nordeste','PB':'Nordeste',
    'PE':'Nordeste','PI':'Nordeste','RN':'Nordeste','SE':'Nordeste',
    'DF':'Centro-Oeste','GO':'Centro-Oeste','MS':'Centro-Oeste','MT':'Centro-Oeste',
    'ES':'Sudeste','MG':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    'PR':'Sul','RS':'Sul','SC':'Sul',
}

# ── Função de carregamento robusto ──────────────────────────────────
ENCODINGS = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']

def carregar_arquivo(caminho):
    """Carrega xlsx ou csv com detecção automática de encoding."""
    ext = caminho.lower().split('.')[-1]
    if ext in ('xlsx', 'xls'):
        # openpyxl cuida do encoding do Excel automaticamente
        df = pd.read_excel(caminho, dtype=str)
        print(f"Lido como Excel (.{ext})")
    else:
        df = None
        for enc in ENCODINGS:
            try:
                df = pd.read_csv(caminho, sep=';', encoding=enc, dtype=str)
                print(f"Lido como CSV — encoding: {enc}")
                break
            except (UnicodeDecodeError, Exception):
                continue
        if df is None:
            raise ValueError("Não foi possível ler o arquivo. Verifique o formato.")
    return df

# ── Carregar ─────────────────────────────────────────────────────────
ARQUIVO = 'armadores_pesca.xlsx'   # altere se necessário

df_raw = carregar_arquivo(ARQUIVO)

print(f"\nTotal de linhas: {len(df_raw)}")
print(f"Colunas: {df_raw.columns.tolist()}")
print("\nPrimeiras linhas:")
df_raw.head(3)

### 3b. Limpeza e normalização das colunas

In [ ]:
# ── Padronizar nomes de colunas (remover espaços/acentos do header) ─
df_raw.columns = (df_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('ascii')
)
print("Colunas padronizadas:", df_raw.columns.tolist())

# ── Identificar colunas relevantes ──────────────────────────────────
# Depois da padronização, as colunas ficam assim:
#   nome___razao_social  |  vinculo_com_o_rgp  |  numero_no_rgp
#   cidade               |  uf                 |  tipo_de_pessoa
#
# Se o seu arquivo tiver nomes diferentes, ajuste aqui:
COL_CIDADE = 'cidade'
COL_UF     = 'uf'
COL_TIPO   = 'tipo_de_pessoa'
COL_VINCULO= 'vinculo_com_o_rgp'

# ── Verificar se colunas existem ────────────────────────────────────
for col in [COL_CIDADE, COL_UF]:
    if col not in df_raw.columns:
        print(f"⚠️  Coluna '{col}' não encontrada. Colunas disponíveis: {df_raw.columns.tolist()}")
        print("   Ajuste as variáveis COL_CIDADE e COL_UF acima.")

# ── Extrair município da coluna cidade ──────────────────────────────
# Formato: "Belém,PA,Brasil" → "Belém"
# Também trata formatos como "Belém" (só nome) ou "Belém - PA"
def extrair_municipio(val):
    if pd.isna(val): return None
    v = str(val).strip()
    if ',' in v:
        return v.split(',')[0].strip()
    if ' - ' in v:
        return v.split(' - ')[0].strip()
    return v

df_raw['municipio'] = df_raw[COL_CIDADE].apply(extrair_municipio)

# ── Limpar UF ────────────────────────────────────────────────────────
df_raw['estado'] = df_raw[COL_UF].str.strip().str.upper()

# ── Filtrar apenas UFs válidas ────────────────────────────────────────
ufs_validas = set(REGIOES.keys())
invalidas = df_raw[~df_raw['estado'].isin(ufs_validas)]['estado'].value_counts()
if len(invalidas):
    print(f"⚠️  UFs não reconhecidas (serão removidas): {invalidas.to_dict()}")

df_clean = df_raw[df_raw['estado'].isin(ufs_validas)].copy()
df_clean['regiao'] = df_clean['estado'].map(REGIOES)

print(f"\nRegistros válidos: {len(df_clean)} (de {len(df_raw)} totais)")
print(f"Estados presentes: {sorted(df_clean['estado'].unique())}")
print("\nDistribuição por região:")
print(df_clean.groupby('regiao').size().sort_values(ascending=False).rename('n'))

## 4. Geocodificação: obter latitude/longitude por município

O dataset **não tem coordenadas** — precisamos geocodificar por nome do município + UF.

**Estratégia eficiente:**
1. Pegar apenas os pares únicos `(municipio, estado)` — evita milhares de chamadas repetidas
2. Geocodificar cada par único (~300–500 municípios distintos no dataset real)
3. Fazer `merge` com o dataframe completo

**Tempo estimado:** ~5–15 minutos para o dataset completo (1 req/segundo para respeitar a API Nominatim).


In [ ]:
from geopy.geocoders import Nominatim

geo = Nominatim(user_agent='cgvis_pescadores_ufrgs', timeout=10)

# ── Municípios únicos ─────────────────────────────────────────────────
pares_unicos = df_clean[['municipio','estado']].drop_duplicates().reset_index(drop=True)
print(f"Pares únicos município+estado a geocodificar: {len(pares_unicos)}")
print("Primeiros 5:", pares_unicos.head().values.tolist())

# ── Carregar cache se existir (evita re-geocodificar ao re-executar) ──
CACHE_FILE = 'geocode_cache.json'
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE) as f:
        cache = json.load(f)
    print(f"Cache carregado: {len(cache)} entradas")
else:
    cache = {}

# ── Geocodificar ──────────────────────────────────────────────────────
erros = []
for i, row in pares_unicos.iterrows():
    mun = str(row['municipio']).strip()
    uf  = str(row['estado']).strip()
    key = f"{mun}|{uf}"
    if key in cache:
        continue
    try:
        loc = geo.geocode(f"{mun}, {uf}, Brasil")
        if loc:
            cache[key] = [round(loc.latitude, 5), round(loc.longitude, 5)]
        else:
            cache[key] = [None, None]
            erros.append(key)
        time.sleep(1.1)  # respeitar limite Nominatim
    except Exception as e:
        cache[key] = [None, None]
        erros.append(key)
        time.sleep(2)

    if (i+1) % 20 == 0:
        with open(CACHE_FILE, 'w') as f: json.dump(cache, f)
        print(f"  Progresso: {i+1}/{len(pares_unicos)} — cache salvo")

# Salvar cache final
with open(CACHE_FILE, 'w') as f: json.dump(cache, f)
print(f"\nGeocodificação concluída!")
print(f"  OK:    {sum(1 for v in cache.values() if v[0] is not None)}")
print(f"  Erros: {len(erros)}")
if erros: print(f"  Municípios sem coordenada: {erros[:10]}")

In [ ]:
# ── Merge das coordenadas com o dataframe ─────────────────────────────
def get_coord(mun, uf, idx):
    key = f"{str(mun).strip()}|{str(uf).strip()}"
    return cache.get(key, [None, None])[idx]

df_clean['latitude']  = df_clean.apply(lambda r: get_coord(r['municipio'],r['estado'],0), axis=1)
df_clean['longitude'] = df_clean.apply(lambda r: get_coord(r['municipio'],r['estado'],1), axis=1)

df = df_clean.dropna(subset=['latitude','longitude']).copy()
df['latitude']  = df['latitude'].astype(float)
df['longitude'] = df['longitude'].astype(float)

TOTAL_GERAL = len(df)
print(f"Registros com coordenadas: {TOTAL_GERAL} (de {len(df_clean)} válidos)")
print(f"Cobertura: {TOTAL_GERAL/len(df_clean)*100:.1f}%")
print("\nDistribuição final por região:")
print(df.groupby('regiao').size().sort_values(ascending=False).rename('pescadores'))

## 5. Pré-cálculo de estatísticas por estado e região

In [ ]:
por_uf  = df.groupby(['estado','regiao']).size().reset_index(name='n')
por_mun = df.groupby(['estado','municipio']).size().reset_index(name='n')
por_reg = df.groupby('regiao').size().reset_index(name='n')

TOTAL = len(df)

# ── Stats por UF ──────────────────────────────────────────────────────
stats_uf = {}
for _, r in por_uf.iterrows():
    uf, reg = r['estado'], r['regiao']
    n_uf    = int(r['n'])
    n_reg   = int(por_reg[por_reg['regiao']==reg]['n'].values[0])
    tops    = por_mun[por_mun['estado']==uf].sort_values('n',ascending=False).head(5)
    tlist   = [[row['municipio'], int(row['n'])] for _, row in tops.iterrows()]
    conc    = tlist[0][1]/n_uf*100 if tlist else 0
    stats_uf[uf] = {
        'n':n_uf, 'regiao':reg, 'n_reg':n_reg,
        'pct_br':  round(n_uf/TOTAL*100, 1),
        'pct_reg': round(n_uf/n_reg*100, 1),
        'tops':    tlist,
        'conc':    round(conc, 1),
        'n_munis': int(df[df['estado']==uf]['municipio'].nunique()),
    }

# ── Stats por região ──────────────────────────────────────────────────
stats_reg = {}
for _, r in por_reg.iterrows():
    reg, n_reg = r['regiao'], int(r['n'])
    ufs_r   = por_uf[por_uf['regiao']==reg].sort_values('n',ascending=False)
    top_ufs = [[row['estado'], int(row['n'])] for _, row in ufs_r.iterrows()]
    top_mun_r = (por_mun[por_mun['estado'].map(REGIOES)==reg]
                 .sort_values('n',ascending=False).head(5))
    top_muns = [[row['municipio'], int(row['n'])] for _, row in top_mun_r.iterrows()]
    vals = [v for _,v in top_ufs]
    stats_reg[reg] = {
        'n':      n_reg,
        'pct_br': round(n_reg/TOTAL*100, 1),
        'n_ufs':  len(ufs_r),
        'top_ufs': top_ufs,
        'top_muns': top_muns,
        'desigualdade': round(np.std(vals)/np.mean(vals)*100, 1) if vals else 0,
        'lider_pct': round(top_ufs[0][1]/n_reg*100, 1) if top_ufs else 0,
    }

print(f"Stats: {len(stats_uf)} estados, {len(stats_reg)} regiões")
print("\nTop 5 estados:")
for uf, s in sorted(stats_uf.items(), key=lambda x: -x[1]['n'])[:5]:
    print(f"  {uf} ({s['regiao']}): {s['n']:,} ({s['pct_br']}% do Brasil)")

## 6. Download do GeoJSON dos estados brasileiros

In [ ]:
GJ_URL = ('https://raw.githubusercontent.com/codeforamerica/click_that_hood'
          '/master/public/data/brazil-states.geojson')
gj = requests.get(GJ_URL, timeout=30).json()

NOME_UF = {
    'Acre':'AC','Amazonas':'AM','Amapá':'AP','Pará':'PA','Rondônia':'RO',
    'Roraima':'RR','Tocantins':'TO','Alagoas':'AL','Bahia':'BA','Ceará':'CE',
    'Maranhão':'MA','Paraíba':'PB','Pernambuco':'PE','Piauí':'PI',
    'Rio Grande do Norte':'RN','Sergipe':'SE','Distrito Federal':'DF',
    'Goiás':'GO','Mato Grosso do Sul':'MS','Mato Grosso':'MT',
    'Espírito Santo':'ES','Minas Gerais':'MG','Rio de Janeiro':'RJ',
    'São Paulo':'SP','Paraná':'PR','Rio Grande do Sul':'RS','Santa Catarina':'SC',
}
NOME_REG = {
    'Acre':'Norte','Amazonas':'Norte','Amapá':'Norte','Pará':'Norte',
    'Rondônia':'Norte','Roraima':'Norte','Tocantins':'Norte',
    'Alagoas':'Nordeste','Bahia':'Nordeste','Ceará':'Nordeste','Maranhão':'Nordeste',
    'Paraíba':'Nordeste','Pernambuco':'Nordeste','Piauí':'Nordeste',
    'Rio Grande do Norte':'Nordeste','Sergipe':'Nordeste',
    'Distrito Federal':'Centro-Oeste','Goiás':'Centro-Oeste',
    'Mato Grosso do Sul':'Centro-Oeste','Mato Grosso':'Centro-Oeste',
    'Espírito Santo':'Sudeste','Minas Gerais':'Sudeste',
    'Rio de Janeiro':'Sudeste','São Paulo':'Sudeste',
    'Paraná':'Sul','Rio Grande do Sul':'Sul','Santa Catarina':'Sul',
}

n_uf_d  = dict(zip(por_uf['estado'],  por_uf['n']))
n_reg_d = dict(zip(por_reg['regiao'], por_reg['n']))

for feat in gj['features']:
    nome = feat['properties'].get('name','')
    uf   = NOME_UF.get(nome,'')
    reg  = NOME_REG.get(nome,'Desconhecido')
    feat['properties'].update({
        'uf':uf, 'regiao':reg,
        'n_estado': int(n_uf_d.get(uf,0)),
        'n_regiao': int(n_reg_d.get(reg,0)),
    })

print(f"GeoJSON: {len(gj['features'])} estados enriquecidos")

## 7. Preparação dos dados de visualização (pontos, heatmap, bbox)

In [ ]:
# Pontos por estado (amostrado — máx 500 por UF para performance)
pts_uf = {}
for uf in df['estado'].unique():
    sub = df[df['estado']==uf].sample(min(500,len(df[df['estado']==uf])),random_state=42)
    pts_uf[uf] = sub[['latitude','longitude','municipio']].values.tolist()

# Pontos por região (amostrado — máx 1500 por região)
pts_reg = {}
for reg in df['regiao'].unique():
    sub = df[df['regiao']==reg].sample(min(1500,len(df[df['regiao']==reg])),random_state=42)
    pts_reg[reg] = sub[['latitude','longitude','municipio','estado']].values.tolist()

# Heatmap por estado (todos os pontos — sem amostragem)
heat_uf = {}
for uf in df['estado'].unique():
    heat_uf[uf] = df[df['estado']==uf][['latitude','longitude']].values.tolist()

# Heatmap por região
heat_reg = {}
for reg in df['regiao'].unique():
    heat_reg[reg] = df[df['regiao']==reg][['latitude','longitude']].values.tolist()

# Heatmap Brasil todo
heat_br = df[['latitude','longitude']].values.tolist()

# Bounding boxes por estado (zoom preciso usando dados reais)
bbox_uf = {}
for uf in df['estado'].unique():
    sub = df[df['estado']==uf]
    pad = 0.3  # graus de padding ao redor dos pontos
    bbox_uf[uf] = [
        float(sub['latitude'].min())  - pad,
        float(sub['longitude'].min()) - pad,
        float(sub['latitude'].max())  + pad,
        float(sub['longitude'].max()) + pad,
    ]

print(f"Pontos preparados: {len(pts_uf)} estados, {len(pts_reg)} regiões")
print(f"Heatmap Brasil: {len(heat_br):,} pontos")

## 8. Geração do mapa HTML interativo

Esta célula serializa todos os dados em JSON e chama `build_map_v3.py` para montar o HTML completo.

O HTML usa **Leaflet.js** + **leaflet.heat** e não precisa de servidor — abre diretamente no navegador.

### O que o mapa inclui:
- Brasil isolado com fundo escuro (sem tiles do mundo)
- Polígonos dos estados coloridos por região (5 cores distintas)
- Clique num estado → isola visualmente, aplica zoom, filtra heatmap e pontos para aquela UF
- Painel lateral com Resumo, Ranking, Insights e Melhorias
- Modal de relatório completo com download `.txt` e cópia para clipboard
- Conclusões e análises embutidas no painel


In [ ]:
import os, json

# Serializar dados para JSON (serão injetados no JS do HTML)
GJ_STR   = json.dumps(gj,       ensure_ascii=False)
S_UF     = json.dumps(stats_uf, ensure_ascii=False)
S_REG    = json.dumps(stats_reg,ensure_ascii=False)
PTS_UF   = json.dumps(pts_uf,   ensure_ascii=False)
PTS_REG  = json.dumps(pts_reg,  ensure_ascii=False)
HEAT_UF  = json.dumps(heat_uf,  ensure_ascii=False)
HEAT_REG = json.dumps(heat_reg, ensure_ascii=False)
HEAT_BR  = json.dumps(heat_br,  ensure_ascii=False)
BBOX_UF  = json.dumps(bbox_uf,  ensure_ascii=False)

# Conclusões e insights embutidos
CONCLUSOES = {
    'geral': [
        "O Nordeste concentra o maior volume de pescadores do Brasil — mais que qualquer outra região, apesar de não ser a mais rica. A pesca artesanal é central para a subsistência dessas comunidades.",
        "O Norte tem a maioria dos seus pescadores em municípios ribeirinhos do interior, não no litoral — revelando que a pesca brasileira não é exclusivamente marítima.",
        "O Sul tem a maior densidade de pescadores por km de litoral. Itajaí/SC e Rio Grande/RS são os dois maiores portos pesqueiros — pesca oceânica e industrial dominam.",
        "O Sudeste combina pesca industrial (Santos, Macaé) com artesanal costeira. RJ e SP concentram a grande maioria da região, revelando forte polarização urbana dos armadores.",
        "O Centro-Oeste representa a menor fatia do total nacional. A pesca pantaneira de Corumbá/MS é a única concentração relevante da região.",
    ],
    'insights_uf': {
        'CE':'Fortaleza concentra mais da metade dos pescadores cearenses — reflexo da capital como hub de pesca industrial e artesanal no Nordeste.',
        'MA':'São Luís domina o estado, mas Cururupu (litoral ocidental) emerge como segundo polo — pesca artesanal de subsistência relevante.',
        'AM':'Manaus concentra a maior parte do estado. A pesca fluvial amazônica se organiza em torno de grandes centros urbanos ribeirinhos.',
        'PA':'Belém lidera com folga. Santarém, 800 km rio acima, é o segundo polo — revelando a vocação pesqueira ao longo do Amazonas.',
        'SC':'Itajaí é o maior porto pesqueiro do Brasil em volume desembarcado. A alta concentração reflete pesca industrial oceânica.',
        'RS':'Rio Grande concentra a maior parte dos gaúchos — polo da pesca oceânica do extremo sul, incluindo alto mar.',
        'RJ':'Macaé aparece como segundo polo fluminense, ligado à plataforma continental e à indústria offshore do petróleo.',
        'SP':'Santos e Guarujá somam mais da metade dos pescadores paulistas — confirmando o litoral central como eixo da pesca no estado.',
        'BA':'Salvador domina a Bahia, mas Ilhéus e Porto Seguro revelam dispersão pelo litoral sul — pesca artesanal tradicional.',
        'RN':'Areia Branca surpreende como 2º polo no RN. Exportação de lagosta e camarão explica a concentração.',
        'PE':'Recife domina, mas Olinda e Cabo de Sto Agostinho mostram que a Grande Recife centraliza toda a pesca pernambucana.',
        'PI':'Parnaíba concentra quase toda a pesca piauiense — apesar do estado ser majoritariamente interiorano.',
        'MS':'Corumbá responde pela maior parte do MS — única saída para o Pantanal e Rio Paraguai, âncora da pesca fluvial do Centro-Oeste.',
    },
    'insights_reg': {
        'Norte':'A maioria dos pescadores do Norte está no interior ribeirinho, não no litoral. Único padrão fluvial dominante do Brasil — reflexo direto da Bacia Amazônica.',
        'Nordeste':'Maior volume absoluto do país. Pesca artesanal é estratégica para segurança alimentar. CE, BA e MA respondem pela maior fatia da região.',
        'Centro-Oeste':'Menor representação nacional. Pesca restrita às bacias do Pantanal e Araguaia-Tocantins. Alta concentração em poucos municípios.',
        'Sudeste':'Combina pesca industrial em portos (Santos, Macaé) e artesanal costeira no litoral fluminense e paulista.',
        'Sul':'Maior densidade relativa de pescadores por km². Itajaí e Rio Grande são referências nacionais em pesca oceânica e processamento industrial.',
    },
    'melhorias': [
        '📊 Série temporal: evolução do número de registros por ano (se disponível no dataset)',
        '🎯 Filtro por tipo: diferenciar Pessoa Física × Pessoa Jurídica e Armador × Proprietário',
        '📈 Correlação IDH: verificar se municípios pesqueiros têm IDH acima/abaixo da média estadual',
        '🌊 Overlay de bacias hidrográficas: mostrar correlação entre rios e concentração no Norte',
        '🔍 Busca por município: campo de texto para localizar um município específico no mapa',
        '📱 Layout responsivo: painel lateral vira modal inferior em telas pequenas',
        '🗓️ Gráfico de barras comparativo animado ao trocar entre regiões',
        '📤 Exportar relatório em PDF com gráficos embutidos',
    ]
}

CONCLUSOES_STR = json.dumps(CONCLUSOES, ensure_ascii=False)
TOTAL_STR = f"{TOTAL:,}".replace(',','.')

# Executar o script gerador do mapa
# Certifique-se que build_map_v3.py está na mesma pasta
if os.path.exists('build_map_v3.py'):
    # Injetar variáveis no escopo e executar
    exec(open('build_map_v3.py').read())
    print("✅ Mapa gerado via build_map_v3.py")
else:
    print("⚠️ build_map_v3.py não encontrado.")
    print("   Faça upload do arquivo na mesma pasta do notebook.")

## 9. Salvar e baixar o mapa

In [ ]:
OUTPUT = 'mapa_v3.html'

if os.path.exists(OUTPUT):
    size_mb = os.path.getsize(OUTPUT) / 1024 / 1024
    print(f"✅ Arquivo gerado: {OUTPUT}  ({size_mb:.1f} MB)")
    print("   Abra no navegador para interagir com o mapa.")
    print("   Não precisa de internet ou servidor — arquivo standalone.")
else:
    print("❌ Arquivo não encontrado. Verifique a célula anterior.")

# No Google Colab: baixar automaticamente
try:
    from google.colab import files
    files.download(OUTPUT)
    print("Download iniciado no navegador!")
except ImportError:
    pass  # fora do Colab — abra manualmente

## 10. Gráfico estático de resumo (para o RELATORIO.md)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

COR_PLOT = {
    'Norte':'#16a34a','Nordeste':'#d97706','Centro-Oeste':'#92400e',
    'Sudeste':'#1d4ed8','Sul':'#7c3aed',
}

resumo = df.groupby('regiao').size().reset_index(name='total').sort_values('total',ascending=False)
top10  = (df.groupby(['estado','regiao']).size().reset_index(name='total')
            .sort_values('total',ascending=False).head(10))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#f8fafc')
fig.suptitle('Distribuição de Armadores de Pesca no Brasil\nFonte: MPA/SERMOP',
             fontsize=13, fontweight='bold', color='#1e293b')

# Por região
cores_reg = [COR_PLOT.get(r,'#6b7280') for r in resumo['regiao']]
bars = axes[0].barh(resumo['regiao'], resumo['total'], color=cores_reg, edgecolor='white', height=0.6)
axes[0].set_xlabel('Armadores registrados', color='#475569')
axes[0].set_title('Por Região', fontweight='bold', color='#1e293b')
axes[0].set_facecolor('#f8fafc')
for i, v in enumerate(resumo['total']):
    axes[0].text(v+max(resumo['total'])*0.01, i, f'{v:,}', va='center', fontsize=10, color='#475569')
axes[0].tick_params(colors='#475569')

# Top 10 estados
cores_top = [COR_PLOT.get(r,'#6b7280') for r in top10['regiao']]
axes[1].barh(top10['estado'], top10['total'], color=cores_top, edgecolor='white', height=0.6)
axes[1].set_xlabel('Armadores registrados', color='#475569')
axes[1].set_title('Top 10 Estados', fontweight='bold', color='#1e293b')
axes[1].set_facecolor('#f8fafc')
for i, v in enumerate(top10['total']):
    axes[1].text(v+max(top10['total'])*0.01, i, f'{v:,}', va='center', fontsize=9, color='#475569')
axes[1].tick_params(colors='#475569')

patches = [mpatches.Patch(color=c, label=r) for r, c in COR_PLOT.items()]
axes[1].legend(handles=patches, title='Região', loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('grafico_resumo_pescadores.png', dpi=150, bbox_inches='tight', facecolor='#f8fafc')
plt.show()
print("Gráfico salvo: grafico_resumo_pescadores.png")